# ⚡ Comic Dialogue & Character Prominence Analyzer
### Module-by-Module Compiler Pipeline & Conversational Social Network Engine

**Course:** Compiler Design Lab / Natural Language Processing  
This notebook presents the complete architecture **module by module**. Each module includes:
1. Architectural explanation and compiler phase analogy.
2. Full Python code implementation.
3. Test inputs and step-by-step console outputs.


## 📦 Environment Setup
Install required external libraries (`plotly`, `networkx`, `pandas`, `openpyxl`).


In [ ]:
!pip install -q plotly networkx pandas openpyxl

import re
import math
import json
from enum import Enum
from typing import Dict, List, Optional, Tuple, Any
from dataclasses import dataclass, field
import pandas as pd
import networkx as nx
import plotly.graph_objects as go

print("✓ All libraries installed and imported successfully!")


---
## 🏷️ Module 1: Speaker Extraction & Attribution
**Compiler Phase:** Lexical Pre-pass / Source Tagging  
Separates dialogue speaker prefixes (e.g. `BATMAN:`, `[SPIDER-MAN]:`) from the spoken text.


In [ ]:
class SpeakerSplitter:
    def __init__(self):
        # Matches 'SPEAKER: line' or '[SPEAKER]: line'
        self.pattern = re.compile(
            r'^\s*(?:\[(?P<bracket_spk>[^\]]+)\]|(?P<speaker>[A-Za-z0-9_\-\.\s]+?))\s*:\s*(?P<dialogue>.*)$'
        )

    def split(self, raw_line: str) -> Tuple[str, str]:
        match = self.pattern.match(raw_line)
        if match:
            speaker = match.group("bracket_spk") or match.group("speaker")
            dialogue = match.group("dialogue")
            return speaker.strip().upper(), dialogue.strip()
        return "ANONYMOUS", raw_line.strip()

# Verification Test
splitter = SpeakerSplitter()
test_lines = [
    "BATMAN: LOOK OUT, BRUCE! WHERE DID HE GO?",
    "[CLINT]: WHERE COULD HE BE, NATASHA?",
    "HAVE A PEANUT."
]

print(f"{'RAW INPUT':<45} | {'SPEAKER':<12} | {'DIALOGUE'}")
print("-" * 85)
for line in test_lines:
    spk, text = splitter.split(line)
    print(f"{line:<45} | {spk:<12} | {text}")


---
## 🧹 Module 2: Dialogue Cleaning & Normalization
**Compiler Phase:** Preprocessing / Normalization  
Repairs hyphenated line wraps (e.g., `sec- tor` ➔ `sector`), collapses multiple spaces/tabs, and removes soft hyphen noise.


In [ ]:
class DialogueCleaner:
    def clean(self, text: str) -> str:
        # 1. Repair broken hyphenated words across wrap boundaries
        text = re.sub(r'(\b[A-Za-z]+)-\s+([A-Za-z]+\b)', r'\1\2', text)
        # 2. Collapse repeated whitespace and tabs
        text = re.sub(r'[ \t]+', ' ', text)
        return text.strip()

cleaner = DialogueCleaner()
sample_dirty = "I JUST STOP-  PED THE   ENTIRE  WORLD    FROM BEING EATEN."
cleaned_result = cleaner.clean(sample_dirty)

print("Before :", sample_dirty)
print("After  :", cleaned_result)


---
## 🔍 Module 3: Comic Lexer & OCR Scan Error Detection
**Compiler Phase:** Lexical Analysis (Scanner)  
Converts character sequences into classified tokens:
`WORD`, `NUMBER`, `CONTRACTION`, `PUNCT`, `TERMINATOR`, `ELLIPSIS`, `INTERROBANG`, `SOUND_FX`, `UNKNOWN`.  
Any stray OCR artifacts (like `~`, `#`) are captured as lexical errors with exact character positions.


In [ ]:
class TokenType(str, Enum):
    WORD = "WORD"
    NUMBER = "NUMBER"
    CONTRACTION = "CONTRACTION"
    PUNCT = "PUNCT"
    TERMINATOR = "TERMINATOR"
    ELLIPSIS = "ELLIPSIS"
    INTERROBANG = "INTERROBANG"
    SOUND_FX = "SOUND_FX"
    UNKNOWN = "UNKNOWN"

SOUND_EFFECTS = {
    "BOOM", "AAARGH", "AARGH", "POW", "BAM", "ZAP", "SMASH",
    "THWIP", "OOF", "UGH", "CRASH", "SNAP", "BANG", "WHOOSH", "HAHAHA"
}

@dataclass
class Token:
    type: TokenType
    value: str
    start: int
    end: int

class ComicLexer:
    def __init__(self):
        self.re_ellipsis = re.compile(r'^(?:\.{3,}|…)')
        self.re_interrobang = re.compile(r'^(?:\?+!+|!+\?+)[\?!]*')
        self.re_contraction = re.compile(r"^[A-Za-z]+['’][A-Za-z]+")
        self.re_word = re.compile(r'^[A-Za-z]+')
        self.re_number = re.compile(r'^\d+')
        self.re_terminator = re.compile(r'^[.?!]')
        self.re_punct = re.compile(r'^[,:;\-—]')

    def tokenize(self, text: str) -> Tuple[List[Token], List[Dict[str, Any]]]:
        tokens = []
        errors = []
        i = 0
        n = len(text)

        while i < n:
            ch = text[i]
            if ch.isspace():
                i += 1
                continue

            sub = text[i:]
            # Ellipsis
            m = self.re_ellipsis.match(sub)
            if m:
                v = m.group(0)
                tokens.append(Token(TokenType.ELLIPSIS, v, i, i + len(v)))
                i += len(v)
                continue

            # Interrobang
            m = self.re_interrobang.match(sub)
            if m:
                v = m.group(0)
                tokens.append(Token(TokenType.INTERROBANG, v, i, i + len(v)))
                i += len(v)
                continue

            # Contraction
            m = self.re_contraction.match(sub)
            if m:
                v = m.group(0)
                tokens.append(Token(TokenType.CONTRACTION, v, i, i + len(v)))
                i += len(v)
                continue

            # Word / Sound FX
            m = self.re_word.match(sub)
            if m:
                v = m.group(0)
                t_type = TokenType.SOUND_FX if v.upper() in SOUND_EFFECTS else TokenType.WORD
                tokens.append(Token(t_type, v, i, i + len(v)))
                i += len(v)
                continue

            # Number
            m = self.re_number.match(sub)
            if m:
                v = m.group(0)
                tokens.append(Token(TokenType.NUMBER, v, i, i + len(v)))
                i += len(v)
                continue

            # Punctuation
            m = self.re_punct.match(sub)
            if m:
                v = m.group(0)
                tokens.append(Token(TokenType.PUNCT, v, i, i + len(v)))
                i += len(v)
                continue

            # Terminator
            m = self.re_terminator.match(sub)
            if m:
                v = m.group(0)
                tokens.append(Token(TokenType.TERMINATOR, v, i, i + len(v)))
                i += len(v)
                continue

            # Stray OCR Error
            errors.append({"symbol": ch, "position": i})
            tokens.append(Token(TokenType.UNKNOWN, ch, i, i + 1))
            i += 1

        return tokens, errors

lexer = ComicLexer()
test_phrase = "BATMAN~ WE HAVE A 10-99 AT SECTOR 4#?!"
tokens, errors = lexer.tokenize(test_phrase)

print(f"Token Stream for: '{test_phrase}'")
print("-" * 60)
for t in tokens:
    print(f"  [{t.type.value:<12}] '{t.value}' (span {t.start}:{t.end})")

if errors:
    print("\n[!] Detected OCR Scan Errors:")
    for err in errors:
        print(f"    Stray symbol '{err['symbol']}' at character index {err['position']}")


---
## ✂️ Module 4: Multi-Sentence Speech Balloon Splitter
**Compiler Phase:** Statement / Clause Boundary Segmenter  
Comic balloons often combine commands, questions, and statements in one bubble (e.g. `"LOOK OUT, BRUCE! WHERE DID HE GO? HE WAS JUST HERE..."`).  
This module decomposes them while protecting abbreviations like `DR.`, `MR.`, `ST.`.


In [ ]:
class BubbleSplitter:
    def __init__(self):
        # Boundary lookbehind for terminator followed by whitespace and capital letter
        self.boundary_re = re.compile(r'(?<=[.!?…])\s+(?=[A-Z0-9"'\[])')
        self.abbr_re = re.compile(r'\b(DR|MR|MRS|MS|CAPT|PROF|GEN|SGT|ST)\.$', re.IGNORECASE)

    def split(self, text: str) -> List[str]:
        raw_parts = self.boundary_re.split(text.strip())
        merged = []
        for part in raw_parts:
            p = part.strip()
            if not p:
                continue
            if merged and self.abbr_re.search(merged[-1]):
                merged[-1] += " " + p
            else:
                merged.append(p)
        return merged if merged else [text.strip()]

splitter_mod = BubbleSplitter()
complex_balloon = "LOOK OUT, BRUCE! WHERE DID HE GO? HE WAS JUST HERE... DR. STRANGE IS COMING!"
clauses = splitter_mod.split(complex_balloon)

print(f"Original Speech Bubble:\n  \"{complex_balloon}\"\n")
print(f"Decomposed into {len(clauses)} distinct clauses:")
for idx, c in enumerate(clauses, 1):
    print(f"  Clause {idx}: \"{c}\"")


---
## ⚖️ Module 5: Context-Free Sentence Type & Grammar Classifier
**Compiler Phase:** Syntactic / Semantic Classification  
Determines utterance structure:
- **Command:** Starts with an imperative verb (e.g., `LOOK`, `WATCH`, `STOP`) or negative directive (`DON'T`).
- **Question:** Contains terminal interrogative mark (`?`, `?!`).
- **Exclamation:** Emphatic exclamation structure (`!`).
- **Statement:** Standard declarative sentence (`.`).


In [ ]:
class SentenceType(str, Enum):
    STATEMENT = "statement"
    QUESTION = "question"
    EXCLAMATION = "exclamation"
    COMMAND = "command"

IMPERATIVE_VERBS = {
    "LOOK", "WATCH", "WAIT", "STOP", "HOLD", "RUN", "GET", "COME",
    "GO", "LISTEN", "STAY", "HELP", "FIRE", "SHOOT", "ATTACK", "DUCK",
    "TAKE", "GIVE", "DROP", "STAND", "MOVE", "SHUT", "OPEN", "TELL"
}

class SentenceClassifier:
    def classify(self, text: str, tokens: List[Token]) -> Tuple[SentenceType, str]:
        t_text = text.strip()
        words = [t.value.upper() for t in tokens if t.type in (TokenType.WORD, TokenType.CONTRACTION)]
        has_qmark = "?" in t_text
        has_exclam = "!" in t_text

        # 1. Imperative Command
        if words and words[0] in IMPERATIVE_VERBS:
            return SentenceType.COMMAND, f"Starts with imperative verb '{words[0]}'"
        if len(words) >= 2 and words[0] in {"DON'T", "DONT"}:
            return SentenceType.COMMAND, "Negative imperative directive"

        # 2. Question
        if has_qmark:
            return SentenceType.QUESTION, "Contains interrogation punctuation (?)"

        # 3. Exclamation
        if has_exclam:
            return SentenceType.EXCLAMATION, "Emphatic terminal exclamation (!)"

        # 4. Statement
        return SentenceType.STATEMENT, "Standard declarative sentence"

classifier = SentenceClassifier()
test_samples = [
    "LOOK OUT, BRUCE!",
    "WHERE COULD HE BE, NATASHA?",
    "I JUST STOPPED THE ENTIRE WORLD FROM BEING EATEN.",
    "BAM! THAT HURT!"
]

print(f"{'UTTERANCE':<50} | {'TYPE':<12} | {'REASON'}")
print("-" * 90)
for sample in test_samples:
    t_tokens, _ = lexer.tokenize(sample)
    s_type, reason = classifier.classify(sample, t_tokens)
    print(f"{sample:<50} | {s_type.value.upper():<12} | {reason}")


---
## 🗄️ Module 6: Symbol Table
**Compiler Phase:** Symbol Table Management  
Maintains unique identifiers, tracks token classes, cumulative counts, and identifies candidate character names.


In [ ]:
class SymbolTable:
    def __init__(self):
        self.table: Dict[str, Dict[str, Any]] = {}

    def insert(self, token_val: str, token_type: str, is_proper_name: bool = False):
        key = token_val.upper()
        if key not in self.table:
            self.table[key] = {
                "name": token_val,
                "type": token_type,
                "count": 1,
                "is_proper_name": is_proper_name
            }
        else:
            self.table[key]["count"] += 1
            if is_proper_name:
                self.table[key]["is_proper_name"] = True

    def to_dataframe(self) -> pd.DataFrame:
        records = [
            {"Token": entry["name"], "Class": entry["type"], "Count": entry["count"], "Proper Name": entry["is_proper_name"]}
            for entry in self.table.values()
        ]
        return pd.DataFrame(records)

sym_table = SymbolTable()
sample_line = "BATMAN AND ROBIN ARE IN GOTHAM CITY."
tokens_st, _ = lexer.tokenize(sample_line)
known_names = {"BATMAN", "ROBIN", "GOTHAM"}

for t in tokens_st:
    is_name = t.value.upper() in known_names
    sym_table.insert(t.value, t.type.value, is_proper_name=is_name)

print("Symbol Table Snapshot:")
display(sym_table.to_dataframe())


---
## 🦸 Module 7: Character Alias Resolution & Universe Mapping
**Compiler Phase:** Semantic Mapping / Alias Binding  
Maps aliases and secret identities (e.g. `THE DARK KNIGHT`, `BRUCE`, `BRUCE WAYNE`) to their canonical character identity (`BATMAN`).


In [ ]:
class AliasResolver:
    def __init__(self, alias_dict: Dict[str, str]):
        self.aliases = {k.upper(): v.upper() for k, v in alias_dict.items()}

    def resolve(self, name: str) -> str:
        clean = name.strip().upper()
        return self.aliases.get(clean, clean)

dc_marvel_aliases = {
    "BATMAN": "BATMAN", "THE BATMAN": "BATMAN", "DARK KNIGHT": "BATMAN", "BRUCE": "BATMAN", "BRUCE WAYNE": "BATMAN",
    "SUPERMAN": "SUPERMAN", "MAN OF STEEL": "SUPERMAN", "CLARK": "SUPERMAN", "CLARK KENT": "SUPERMAN",
    "ROBIN": "ROBIN", "BOY WONDER": "ROBIN", "DICK GRAYSON": "ROBIN",
    "IRON MAN": "IRON MAN", "TONY": "IRON MAN", "TONY STARK": "IRON MAN",
    "BLACK WIDOW": "BLACK WIDOW", "NATASHA": "BLACK WIDOW", "NATASHA ROMANOFF": "BLACK WIDOW",
    "GREEN ARROW": "GREEN ARROW", "OLIVER": "GREEN ARROW", "OLLIE": "GREEN ARROW"
}

resolver = AliasResolver(dc_marvel_aliases)
candidates = ["Bruce", "The Dark Knight", "Ollie", "Tony Stark", "Natasha"]

print(f"{'ALIAS / MENTION':<20} -> {'CANONICAL CHARACTER IDENTITY'}")
print("-" * 50)
for c in candidates:
    print(f"{c:<20} -> {resolver.resolve(c)}")


---
## 🏆 Module 8: Character Popularity & Prominence Engine
**Compiler Phase:** Semantic Metrics & Scoring  
Evaluates character prominence across dialogue turns using the formula:
$$\text{Popularity}(C) = 1.0 \cdot S + 1.5 \cdot A + 0.8 \cdot M + 0.2 \cdot T$$
Where:
- $S$: Lines Spoken
- $A$: Times Directly Addressed (Vocative callouts)
- $M$: Times Mentioned
- $T$: Dramatic Tone Cues (Shouting, Interrogations)


In [ ]:
@dataclass
class CharacterStats:
    canonical_name: str
    spoken_count: int = 0
    addressed_count: int = 0
    mentioned_count: int = 0
    tone_count: int = 0
    addressed_others: Dict[str, int] = field(default_factory=dict)

    @property
    def popularity_score(self) -> float:
        return round(
            1.0 * self.spoken_count +
            1.5 * self.addressed_count +
            0.8 * self.mentioned_count +
            0.2 * self.tone_count,
            2
        )

class PopularityEngine:
    def __init__(self, resolver: AliasResolver):
        self.resolver = resolver
        self.registry: Dict[str, CharacterStats] = {}

    def get_or_create(self, name: str) -> CharacterStats:
        canon = self.resolver.resolve(name)
        if canon not in self.registry:
            self.registry[canon] = CharacterStats(canonical_name=canon)
        return self.registry[canon]

    def record_speech(self, speaker: str, text: str):
        spk_obj = self.get_or_create(speaker)
        spk_obj.spoken_count += 1
        if "!" in text or "?" in text:
            spk_obj.tone_count += 1

    def record_address(self, speaker: str, target: str):
        spk_canon = self.resolver.resolve(speaker)
        target_obj = self.get_or_create(target)
        target_obj.addressed_count += 1
        spk_obj = self.get_or_create(speaker)
        spk_obj.addressed_others[target_obj.canonical_name] = (
            spk_obj.addressed_others.get(target_obj.canonical_name, 0) + 1
        )

    def record_mention(self, target: str):
        target_obj = self.get_or_create(target)
        target_obj.mentioned_count += 1

    def compute_leaderboard(self) -> pd.DataFrame:
        rankings = sorted(self.registry.values(), key=lambda c: c.popularity_score, reverse=True)
        rows = [
            {
                "Rank": i,
                "Character": c.canonical_name,
                "Popularity Score": c.popularity_score,
                "Spoken (S)": c.spoken_count,
                "Addressed (A)": c.addressed_count,
                "Mentioned (M)": c.mentioned_count,
                "Tone Bonus (T)": c.tone_count
            }
            for i, c in enumerate(rankings, 1)
        ]
        return pd.DataFrame(rows)

pop_engine = PopularityEngine(resolver)

# Simulate interactions
pop_engine.record_speech("BATMAN", "LOOK OUT, BRUCE! WHERE DID HE GO?")
pop_engine.record_address("BATMAN", "BRUCE")
pop_engine.record_speech("CLINT", "WHERE COULD HE BE, NATASHA?")
pop_engine.record_address("CLINT", "NATASHA")
pop_engine.record_speech("GREEN ARROW", "I JUST STOPPED THE ENTIRE WORLD, DINAH.")
pop_engine.record_mention("BRUCE WAYNE")

print("Character Popularity Leaderboard:")
display(pop_engine.compute_leaderboard())


---
## 🕸️ Module 9: Conversational Social Network & Centrality Graph
**Compiler Phase:** Graph Analysis & Data Flow Visualization  
Constructs a **Directed Social Interaction Network** ($A \rightarrow B$) where an edge represents character $A$ calling out to character $B$.  
Calculates:
- **Degree Centrality:** Total conversational connections
- **In-Degree Centrality:** Who gets addressed by others the most
- **Betweenness Centrality:** Who acts as an interaction bridge
- **PageRank:** Network prominence and influence


In [ ]:
class ConversationalNetworkAnalyzer:
    def __init__(self, registry: Dict[str, CharacterStats]):
        self.registry = registry

    def build_network_metrics(self) -> Tuple[nx.DiGraph, pd.DataFrame]:
        G = nx.DiGraph()
        rankings = sorted(self.registry.values(), key=lambda c: c.popularity_score, reverse=True)

        for c in rankings:
            G.add_node(c.canonical_name, score=c.popularity_score)
            for target, count in c.addressed_others.items():
                G.add_edge(c.canonical_name, target, weight=count)

        n = len(G.nodes)
        deg_c = nx.degree_centrality(G) if n > 1 else {k: 0.0 for k in G.nodes}
        in_deg_c = nx.in_degree_centrality(G) if n > 1 else {k: 0.0 for k in G.nodes}
        out_deg_c = nx.out_degree_centrality(G) if n > 1 else {k: 0.0 for k in G.nodes}
        bet_c = nx.betweenness_centrality(G, weight="weight") if n > 1 else {k: 0.0 for k in G.nodes}
        try:
            pr = nx.pagerank(G, weight="weight") if n > 1 else {k: 1.0 for k in G.nodes}
        except:
            pr = {k: 1.0 / max(1, n) for k in G.nodes}

        rows = []
        for c in rankings:
            name = c.canonical_name
            rows.append({
                "Character": name,
                "Score": c.popularity_score,
                "In-Degree": G.in_degree(name),
                "Out-Degree": G.out_degree(name),
                "Degree Centrality": round(deg_c.get(name, 0.0), 4),
                "In-Degree Centrality": round(in_deg_c.get(name, 0.0), 4),
                "Betweenness Centrality": round(bet_c.get(name, 0.0), 4),
                "PageRank": round(pr.get(name, 0.0), 4)
            })
        return G, pd.DataFrame(rows)

    def plot_interactive_graph(self, G: nx.DiGraph, metrics_df: pd.DataFrame):
        pos = nx.circular_layout(G) if len(G.nodes) <= 12 else nx.spring_layout(G, seed=42)
        edge_x, edge_y, annotations = [], [], []

        for src, dst, data in G.edges(data=True):
            x0, y0 = float(pos[src][0]), float(pos[src][1])
            x1, y1 = float(pos[dst][0]), float(pos[dst][1])
            weight = data.get("weight", 1)
            edge_x.extend([x0, x1, None])
            edge_y.extend([y0, y1, None])

            dx, dy = x1 - x0, y1 - y0
            if math.hypot(dx, dy) > 0.05:
                annotations.append(dict(
                    ax=float(x0 + 0.60 * dx), ay=float(y0 + 0.60 * dy),
                    x=float(x0 + 0.75 * dx), y=float(y0 + 0.75 * dy),
                    xref="x", yref="y", axref="x", ayref="y",
                    showarrow=True, arrowhead=2, arrowsize=1.2, arrowwidth=2.0,
                    arrowcolor="rgba(148, 163, 184, 0.75)"
                ))

        edge_trace = go.Scatter(
            x=edge_x, y=edge_y,
            line=dict(width=1.5, color="rgba(148, 163, 184, 0.45)"),
            hoverinfo="none", mode="lines"
        )

        scores = metrics_df["Score"].tolist()
        min_s, max_s = min(scores), max(scores)
        s_range = max(1.0, max_s - min_s)
        sizes = [20.0 + ((s - min_s) / s_range) * 35.0 for s in scores]

        node_trace = go.Scatter(
            x=[float(pos[row["Character"]][0]) for _, row in metrics_df.iterrows()],
            y=[float(pos[row["Character"]][1]) for _, row in metrics_df.iterrows()],
            mode="markers+text",
            text=metrics_df["Character"].tolist(),
            textposition="top center",
            marker=dict(
                showscale=True,
                colorscale="Viridis",
                reversescale=True,
                color=metrics_df["In-Degree Centrality"].tolist(),
                size=sizes,
                colorbar=dict(thickness=14, title=dict(text="In-Degree Centrality", side="right")),
                line=dict(width=2, color="#FFFFFF")
            ),
            hovertext=[
                f"<b>{r['Character']}</b><br>Score: {r['Score']}<br>Addressed by others: {r['In-Degree']}<br>Addresses others: {r['Out-Degree']}"
                for _, r in metrics_df.iterrows()
            ],
            hoverinfo="text"
        )

        fig = go.Figure(
            data=[edge_trace, node_trace],
            layout=go.Layout(
                title="Directed Conversational Interaction Network",
                showlegend=False,
                hovermode="closest",
                margin=dict(b=20, l=20, r=20, t=50),
                annotations=annotations,
                xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[-1.35, 1.35]),
                yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[-1.35, 1.35]),
                height=550,
                template="plotly_dark"
            )
        )
        return fig

# Run Network Module
net_mod = ConversationalNetworkAnalyzer(pop_engine.registry)
G, net_df = net_mod.build_network_metrics()

print("Social Network Centrality Metrics:")
display(net_df)

fig_net = net_mod.plot_interactive_graph(G, net_df)
fig_net.show()


---
## 📄 Module 10: Academic Export Suite (CSV, JSON, LaTeX)
**Compiler Phase:** Code Generation / Target Output  
Exports analytics into reproducible academic formats:
1. **CSV:** Spreadsheet analysis
2. **JSON:** Full token and character logs
3. **LaTeX (`booktabs`):** Direct copy-paste into IEEE/ACM research papers


In [ ]:
class ExportSuite:
    @staticmethod
    def export_csv(df: pd.DataFrame, filename: str = "character_rankings.csv"):
        df.to_csv(filename, index=False)
        print(f"✓ Saved CSV export to: {filename}")

    @staticmethod
    def export_json(data: Dict[str, Any], filename: str = "analytics_report.json"):
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
        print(f"✓ Saved JSON export to: {filename}")

    @staticmethod
    def export_latex(df: pd.DataFrame) -> str:
        latex_str = df.to_latex(
            index=False,
            caption="Comic Dialogue Character Popularity and Conversational Centrality",
            label="tab:character_prominence"
        )
        return latex_str

# Run Export
ExportSuite.export_csv(net_df, "colab_character_rankings.csv")
latex_code = ExportSuite.export_latex(net_df)

print("\n=== READY-TO-PASTE LATEX TABLE CODE ===")
print(latex_code)
